In [7]:
import sys
!{sys.executable} -m pip install -q pydantic-ai-slim openai mcp-server-time "fastmcp-slim[client]"

In [4]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("groq-api-key")

os.environ["GROQ_API_KEY"]   = secret_value_0
os.environ["BASE_URL"]       = "https://api.groq.com/openai/v1"
os.environ["OPENAI_API_KEY"] = secret_value_0

print("Config ready. BASE_URL:", os.environ["BASE_URL"])

Config ready. BASE_URL: https://api.groq.com/openai/v1


In [5]:
from openai import OpenAI

client = OpenAI(
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

models = client.models.list()
print("Connected! Available models:")
for m in models.data[:5]:
    print(" •", m.id)

Connected! Available models:
 • openai/gpt-oss-120b
 • openai/gpt-oss-20b
 • canopylabs/orpheus-v1-english
 • meta-llama/llama-prompt-guard-2-86m
 • groq/compound


In [6]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

provider = OpenAIProvider(
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

agent_model = OpenAIChatModel("llama-3.3-70b-versatile", provider=provider)

print("Agent model ready!")

Agent model ready!


In [8]:
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset
from fastmcp.client.transports import StdioTransport

time_server = MCPToolset(
    StdioTransport(
        command="python",
        args=["-m", "mcp_server_time", "--local-timezone=America/New_York"],
    )
)

agent = Agent(
    model=agent_model,
    toolsets=[time_server],
    system_prompt=(
        "You MUST use the get_current_time tool to answer any question about the current date or time. "
        "Never say you don't have access to the time. Always call the tool first, then respond."
    )
)

print("Agent with MCP time server ready!")

Agent with MCP time server ready!


In [11]:
async def run_async(prompt: str) -> str:
    async with agent.run_mcp_servers():
        result = await agent.run(prompt)
        return result.output

In [12]:
answer = await run_async("What's the date today?")
print(answer)

The date today is 2026-05-31.


In [13]:
import sys
!{sys.executable} -m pip install -q mcp-server-fetch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 1.8 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
a2a-sdk 0.3.26 requires httpx>=0.28.1, but you have httpx 0.27.2 which is incompatible.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.27.2 which is incompatible.
google-genai 1.68.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.


In [14]:
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset
from fastmcp.client.transports import StdioTransport

time_server = MCPToolset(
    StdioTransport(
        command="python",
        args=["-m", "mcp_server_time", "--local-timezone=America/New_York"],
    )
)

fetch_server = MCPToolset(
    StdioTransport(
        command="python",
        args=["-m", "mcp_server_fetch"],
    )
)

agent = Agent(
    model=agent_model,
    toolsets=[time_server, fetch_server],
    system_prompt=(
        "You are a helpful agent with access to two tools: "
        "get_current_time for date/time questions, and fetch for retrieving web content. "
        "Always use the appropriate tool when needed."
    )
)

print("Agent with time + fetch MCP servers ready!")

Agent with time + fetch MCP servers ready!


In [16]:
answer = await run_async("Fetch the content from https://example.com and summarize it.")
print(answer)

The content of https://example.com is a simple webpage stating that the domain is for use in documentation examples and should not be used in operations. It also provides a link to learn more about the domain.


In [17]:
answer = await run_async(
    "What is today's date and time? Also fetch https://example.com and give me a one line summary."
)
print(answer)

The current date and time in New York is Sunday, May 31, 2026, 2:17 AM. The website https://example.com is a domain for use in documentation examples.
